In [1]:
#Importing libraries
import numpy as np
import pandas as pd
import sys

In [2]:
# Confirming the environment variales
print("Python:", sys.version.split()[0])
print("Numpy:", np.__version__)
print("Pandas:", pd.__version__)

Python: 3.12.10
Numpy: 2.5.3
Pandas: 3.0.6


In [3]:
# Import path
from pathlib import Path


In [4]:
RAW = Path("../data/raw")
PROCESSSED = Path("../data/processed")

In [5]:
for file in RAW.glob("*.csv"):
    print(file.name)

Bookings_2023.csv
Bookings_2024.csv
Bookings_2025.csv
Packages.csv


In [6]:
print(RAW.exists())

True


In [7]:
# Working with Pandas to work on the data
# Loading the data using pandas

df_2023 = pd.read_csv(RAW / "Bookings_2023.csv")
df_2024 = pd.read_csv(RAW / "Bookings_2024.csv")
df_2025 = pd.read_csv(RAW / "Bookings_2025.csv")

In [8]:
#Merging
df = pd.concat([df_2023, df_2024, df_2025], ignore_index=True)
df.shape
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 13927 entries, 0 to 13926
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   BookingID        13927 non-null  str    
 1   BookingDate      13927 non-null  str    
 2   TravelStartDate  13927 non-null  str    
 3   CustomerID       13927 non-null  str    
 4   PackageID        13927 non-null  str    
 5   AgentID          12827 non-null  str    
 6   Adults           13927 non-null  int64  
 7   Children         13927 non-null  int64  
 8   UnitPriceUSD     13927 non-null  int64  
 9   DiscountPct      13510 non-null  float64
 10  Status           13927 non-null  str    
 11  PaymentMethod    13927 non-null  str    
dtypes: float64(1), int64(3), str(8)
memory usage: 2.1 MB


,BookingID,BookingDate,TravelStartDate,CustomerID,PackageID,AgentID,Adults,Children,UnitPriceUSD,DiscountPct,Status,PaymentMethod
0,BK-2023-00001,2023-01-01,2023-02-20,C01696,PK06,AG10,2,0,1250,0.00,Completed,Bank Transfer
1,BK-2023-00002,2023-01-01,2023-07-07,C01980,PK09,AG02,1,0,2990,0.00,Completed,Card
2,BK-2023-00003,2023-01-01,2023-08-27,C01992,PK15,AG08,2,3,4485,0.00,Completed,PayPal
3,BK-2023-00004,2023-01-01,2023-03-27,C02537,PK06,NaN,1,0,1250,0.00,Completed,Card
4,BK-2023-00005,2023-01-01,2023-02-08,C03846,PK01,NaN,1,0,420,0.15,Completed,Bank Transfer


In [9]:
# Data Cleaning
# Checking duplicates
df["BookingID"].duplicated().sum()


np.int64(37)

In [10]:
# Remove duplicates bookings
df = df.drop_duplicates(subset="BookingID",keep="first")

In [11]:
# Missing Values
df.isna().sum()

BookingID             0
BookingDate           0
TravelStartDate       0
CustomerID            0
PackageID             0
AgentID            1100
Adults                0
Children              0
UnitPriceUSD          0
DiscountPct         417
Status                0
PaymentMethod         0
dtype: int64

In [12]:
# Clean missing values
df["AgentID"] = df["AgentID"].fillna("Unknown")
df["DiscountPct"] = df["DiscountPct"].fillna(0)

In [13]:
# Check data quality on status column
df["Status"].value_counts(dropna=False)

Status
Completed     11387
Cancelled       851
completed       401
Completed       375
Canceled        346
COMPLETED       245
Confirmed       161
Cancelled        47
cancelled        41
CANCELLED        27
CONFIRMED         4
Confirmed         3
confirmed         2
Name: count, dtype: int64

In [14]:
df["Status"].unique()

<ArrowStringArray>
[ 'Completed',  'cancelled',  'COMPLETED',  'CANCELLED',  'Cancelled',
  'completed',   'Canceled', 'Cancelled ', 'Completed ',  'Confirmed',
  'CONFIRMED',  'confirmed', 'Confirmed ']
Length: 13, dtype: str

In [15]:
# Cleaning the Status column
df["Status"] = df["Status"].astype("string").str.strip().str.lower().replace({"canceled": "cancelled"})

In [16]:
# Convert the column to datetime objects first

df["BookingDate"] = pd.to_datetime(df["BookingDate"], errors="coerce")
df["TravelStartDate"] = pd.to_datetime(df["TravelStartDate"], errors="coerce")

# Extract data from booking date column
df["BookingYear"] = df["BookingDate"].dt.year
df["BookingMonth"] = df["BookingDate"].dt.month

In [17]:
# Finding data using position in pandas
df.iloc[1]


BookingID                BK-2023-00002
BookingDate        2023-01-01 00:00:00
TravelStartDate    2023-07-07 00:00:00
CustomerID                      C01980
PackageID                         PK09
AgentID                           AG02
Adults                               1
Children                             0
UnitPriceUSD                      2990
DiscountPct                        0.0
Status                       completed
PaymentMethod                     Card
BookingYear                       2023
BookingMonth                         1
Name: 1, dtype: object

In [18]:
df.iloc[1:3]

,BookingID,BookingDate,TravelStartDate,CustomerID,PackageID,AgentID,Adults,Children,UnitPriceUSD,DiscountPct,Status,PaymentMethod,BookingYear,BookingMonth
1,BK-2023-00002,2023-01-01,2023-07-07,C01980,PK09,AG02,1,0,2990,0.0,completed,Card,2023,1
2,BK-2023-00003,2023-01-01,2023-08-27,C01992,PK15,AG08,2,3,4485,0.0,completed,PayPal,2023,1


In [19]:
df.iloc[1:6, 3:6]

,CustomerID,PackageID,AgentID
1,C01980,PK09,AG02
2,C01992,PK15,AG08
3,C02537,PK06,Unknown
4,C03846,PK01,Unknown
5,C04189,PK02,Unknown


In [20]:
df.loc[:, "BookingID"]

0        BK-2023-00001
1        BK-2023-00002
2        BK-2023-00003
3        BK-2023-00004
4        BK-2023-00005
             ...      
13922    BK-2025-05430
13923    BK-2025-05431
13924    BK-2025-05432
13925    BK-2025-05433
13926    BK-2025-05434
Name: BookingID, Length: 13890, dtype: str

In [21]:
df.loc[df["UnitPriceUSD"] > 12000, ["BookingID", "BookingYear", "UnitPriceUSD"]].to_csv("above 1200")

In [22]:
df.loc[df["PaymentMethod"] == "M-Pesa"].to_csv("mpesa_payment")

In [23]:
# Numpy
df.head()

,BookingID,BookingDate,TravelStartDate,CustomerID,PackageID,AgentID,Adults,Children,UnitPriceUSD,DiscountPct,Status,PaymentMethod,BookingYear,BookingMonth
0,BK-2023-00001,2023-01-01,2023-02-20,C01696,PK06,AG10,2,0,1250,0.00,completed,Bank Transfer,2023,1
1,BK-2023-00002,2023-01-01,2023-07-07,C01980,PK09,AG02,1,0,2990,0.00,completed,Card,2023,1
2,BK-2023-00003,2023-01-01,2023-08-27,C01992,PK15,AG08,2,3,4485,0.00,completed,PayPal,2023,1
3,BK-2023-00004,2023-01-01,2023-03-27,C02537,PK06,Unknown,1,0,1250,0.00,completed,Card,2023,1
4,BK-2023-00005,2023-01-01,2023-02-08,C03846,PK01,Unknown,1,0,420,0.15,completed,Bank Transfer,2023,1


In [24]:
df["Travellers"] = df["Adults"] + df["Children"]
df.head()

,BookingID,BookingDate,TravelStartDate,CustomerID,PackageID,AgentID,Adults,Children,UnitPriceUSD,DiscountPct,Status,PaymentMethod,BookingYear,BookingMonth,Travellers
0,BK-2023-00001,2023-01-01,2023-02-20,C01696,PK06,AG10,2,0,1250,0.00,completed,Bank Transfer,2023,1,2
1,BK-2023-00002,2023-01-01,2023-07-07,C01980,PK09,AG02,1,0,2990,0.00,completed,Card,2023,1,1
2,BK-2023-00003,2023-01-01,2023-08-27,C01992,PK15,AG08,2,3,4485,0.00,completed,PayPal,2023,1,5
3,BK-2023-00004,2023-01-01,2023-03-27,C02537,PK06,Unknown,1,0,1250,0.00,completed,Card,2023,1,1
4,BK-2023-00005,2023-01-01,2023-02-08,C03846,PK01,Unknown,1,0,420,0.15,completed,Bank Transfer,2023,1,1


In [25]:
df["GrossValue"] = df["UnitPriceUSD"]
df.head()

,BookingID,BookingDate,TravelStartDate,CustomerID,PackageID,AgentID,Adults,Children,UnitPriceUSD,DiscountPct,Status,PaymentMethod,BookingYear,BookingMonth,Travellers,GrossValue
0,BK-2023-00001,2023-01-01,2023-02-20,C01696,PK06,AG10,2,0,1250,0.00,completed,Bank Transfer,2023,1,2,1250
1,BK-2023-00002,2023-01-01,2023-07-07,C01980,PK09,AG02,1,0,2990,0.00,completed,Card,2023,1,1,2990
2,BK-2023-00003,2023-01-01,2023-08-27,C01992,PK15,AG08,2,3,4485,0.00,completed,PayPal,2023,1,5,4485
3,BK-2023-00004,2023-01-01,2023-03-27,C02537,PK06,Unknown,1,0,1250,0.00,completed,Card,2023,1,1,1250
4,BK-2023-00005,2023-01-01,2023-02-08,C03846,PK01,Unknown,1,0,420,0.15,completed,Bank Transfer,2023,1,1,420


In [26]:
df["DiscAmount"] = df["GrossValue"] * df["DiscountPct"]
df.head()

,BookingID,BookingDate,TravelStartDate,CustomerID,PackageID,AgentID,Adults,Children,UnitPriceUSD,DiscountPct,Status,PaymentMethod,BookingYear,BookingMonth,Travellers,GrossValue,DiscAmount
0,BK-2023-00001,2023-01-01,2023-02-20,C01696,PK06,AG10,2,0,1250,0.00,completed,Bank Transfer,2023,1,2,1250,0.0
1,BK-2023-00002,2023-01-01,2023-07-07,C01980,PK09,AG02,1,0,2990,0.00,completed,Card,2023,1,1,2990,0.0
2,BK-2023-00003,2023-01-01,2023-08-27,C01992,PK15,AG08,2,3,4485,0.00,completed,PayPal,2023,1,5,4485,0.0
3,BK-2023-00004,2023-01-01,2023-03-27,C02537,PK06,Unknown,1,0,1250,0.00,completed,Card,2023,1,1,1250,0.0
4,BK-2023-00005,2023-01-01,2023-02-08,C03846,PK01,Unknown,1,0,420,0.15,completed,Bank Transfer,2023,1,1,420,63.0


In [27]:
df["NetValue"] = df["GrossValue"] -  df["DiscAmount"]
df.head()

,BookingID,BookingDate,TravelStartDate,CustomerID,PackageID,AgentID,Adults,Children,UnitPriceUSD,DiscountPct,Status,PaymentMethod,BookingYear,BookingMonth,Travellers,GrossValue,DiscAmount,NetValue
0,BK-2023-00001,2023-01-01,2023-02-20,C01696,PK06,AG10,2,0,1250,0.00,completed,Bank Transfer,2023,1,2,1250,0.0,1250.0
1,BK-2023-00002,2023-01-01,2023-07-07,C01980,PK09,AG02,1,0,2990,0.00,completed,Card,2023,1,1,2990,0.0,2990.0
2,BK-2023-00003,2023-01-01,2023-08-27,C01992,PK15,AG08,2,3,4485,0.00,completed,PayPal,2023,1,5,4485,0.0,4485.0
3,BK-2023-00004,2023-01-01,2023-03-27,C02537,PK06,Unknown,1,0,1250,0.00,completed,Card,2023,1,1,1250,0.0,1250.0
4,BK-2023-00005,2023-01-01,2023-02-08,C03846,PK01,Unknown,1,0,420,0.15,completed,Bank Transfer,2023,1,1,420,63.0,357.0


In [28]:
# Filter/classif Trvaelers by their booking sizes
df["BookingCategory"] = np.where(
    df["Travellers"] >= 4,
    "Kubwa", "Ndogo"
)
df.head()

,BookingID,BookingDate,TravelStartDate,CustomerID,PackageID,AgentID,Adults,Children,UnitPriceUSD,DiscountPct,Status,PaymentMethod,BookingYear,BookingMonth,Travellers,GrossValue,DiscAmount,NetValue,BookingCategory
0,BK-2023-00001,2023-01-01,2023-02-20,C01696,PK06,AG10,2,0,1250,0.00,completed,Bank Transfer,2023,1,2,1250,0.0,1250.0,Ndogo
1,BK-2023-00002,2023-01-01,2023-07-07,C01980,PK09,AG02,1,0,2990,0.00,completed,Card,2023,1,1,2990,0.0,2990.0,Ndogo
2,BK-2023-00003,2023-01-01,2023-08-27,C01992,PK15,AG08,2,3,4485,0.00,completed,PayPal,2023,1,5,4485,0.0,4485.0,Kubwa
3,BK-2023-00004,2023-01-01,2023-03-27,C02537,PK06,Unknown,1,0,1250,0.00,completed,Card,2023,1,1,1250,0.0,1250.0,Ndogo
4,BK-2023-00005,2023-01-01,2023-02-08,C03846,PK01,Unknown,1,0,420,0.15,completed,Bank Transfer,2023,1,1,420,63.0,357.0,Ndogo


In [29]:
#
df["DiscountCategory"] = np.select(
    [
        df["DiscountPct"] == 0,
        df["DiscountPct"] < 0.10,
        df["DiscountPct"] >= 0.10
    ],
    [
        "No Discount",
        "Low Discount",
        "High Discount"
    ],
    default="Other"
)
df.head()

,BookingID,BookingDate,TravelStartDate,CustomerID,PackageID,AgentID,Adults,Children,UnitPriceUSD,DiscountPct,Status,PaymentMethod,BookingYear,BookingMonth,Travellers,GrossValue,DiscAmount,NetValue,BookingCategory,DiscountCategory
0,BK-2023-00001,2023-01-01,2023-02-20,C01696,PK06,AG10,2,0,1250,0.00,completed,Bank Transfer,2023,1,2,1250,0.0,1250.0,Ndogo,No Discount
1,BK-2023-00002,2023-01-01,2023-07-07,C01980,PK09,AG02,1,0,2990,0.00,completed,Card,2023,1,1,2990,0.0,2990.0,Ndogo,No Discount
2,BK-2023-00003,2023-01-01,2023-08-27,C01992,PK15,AG08,2,3,4485,0.00,completed,PayPal,2023,1,5,4485,0.0,4485.0,Kubwa,No Discount
3,BK-2023-00004,2023-01-01,2023-03-27,C02537,PK06,Unknown,1,0,1250,0.00,completed,Card,2023,1,1,1250,0.0,1250.0,Ndogo,No Discount
4,BK-2023-00005,2023-01-01,2023-02-08,C03846,PK01,Unknown,1,0,420,0.15,completed,Bank Transfer,2023,1,1,420,63.0,357.0,Ndogo,High Discount


In [30]:
# Business Questions
# Is the Business Growing?
# Year over Year Analysis
# Metrics to use --> Travelers, grwth/clients no. growth
#                 --> financial growth over the three years
#                 -->  Number of bookings across the three years to assess growth

BusinessGrowth = (
    df.groupby("BookingYear").agg
    (
        TotalNetValue = ("NetValue", "sum"),
        Travellers_served = ("Travellers", "sum"),
        Booking_made = ("BookingID", "nunique")
    )
    .reset_index()
)

BusinessGrowth.head()

,BookingYear,TotalNetValue,Travellers_served,Booking_made
0,2023,10111891.50,13540,3867
1,2024,13727948.00,15905,4589
2,2025,18310765.25,18955,5434


In [31]:
# 2. Which packages perform best

Best_performing_package = (
    df.groupby("PackageID").agg
    (
        TotalNetValue = ("GrossValue", "sum"),
        Travellers_served = ("Travellers", "sum"),
        Booking_made = ("BookingID", "nunique")
    )
    .reset_index()
    
)

Best_packages = Best_performing_package.sort_values(by="TotalNetValue", ascending=False)

Best_packages.head()

,PackageID,TotalNetValue,Travellers_served,Booking_made
11,PK12,6253025,3850,1101
12,PK13,6085400,2851,828
16,PK17,5057095,1484,435
15,PK16,3529125,2218,647
13,PK14,3334230,2502,686


In [39]:
# Merging the packages table into the best-performing tables fine the top 3 performing
# To load and merge the packages CSV

packages = pd.read_csv(RAW / "Packages.csv")
# packages.head()

packages_merged = packages.merge(
    Best_packages,
    on="PackageID",
    how="inner"
 
)
#print(packages_merged.columns)

#print(packages_merged.head(20))

# Rank by total booking value
top3_packages = packages_merged.sort_values("TotalNetValue", ascending=False).head(3)

print(top3_packages)


   PackageID                       PackageName Category  DurationDays  \
11      PK12  Mara Great Migration Luxury Camp   Luxury             5   
12      PK13        Serengeti Migration Fly-In   Luxury             6   
16      PK17    Ultimate East Africa Honeymoon   Luxury            12   

   PrimaryParkID  BasePriceUSD  CostPerPaxUSD  LaunchDate  TotalNetValue  \
11           P01          4800           2750  2018-01-01        6253025   
12           P06          6200           3600  2019-01-01        6085400   
16           P06          9800           5900  2022-02-14        5057095   

    Travellers_served  Booking_made  
11               3850          1101  
12               2851           828  
16               1484           435  


In [35]:
# 3. Where does booking value come from?

total_value = packages_merged["TotalNetValue"].sum()

by_category = (
    packages_merged.groupby("Category", as_index=False)
    .agg(
        TotalNetValue=("TotalNetValue", "sum"),
        Packages=("PackageID", "nunique"),
        Bookings=("Booking_made", "sum"),
        Travellers=("Travellers_served", "sum"),
    )
)
by_category["ValueSharePct"] = (
    by_category["TotalNetValue"] / total_value * 100
)
by_category.sort_values("TotalNetValue", ascending=False)

,Category,TotalNetValue,Packages,Bookings,Travellers,ValueSharePct
1,Luxury,30615650,7,4877,17155,69.917285
2,Mid-Range,10702390,6,4780,16640,24.441162
0,Budget,2470345,5,4233,14605,5.641553


In [38]:
# Use NumPy to reshape the results and create a pivot table

# Convert the top-three numerical results to a NumPy array
top_3_values = top3_packages[
    ["TotalNetValue", "Booking_made"]
].to_numpy(dtype=float)

# Flatten the values, then reshape them into rows of:
# [total_booking_value, booking_count]
top_3_reshaped = top_3_values.ravel().reshape(len(top3_packages), 2)

print(top_3_reshaped)

[[6.253025e+06 1.101000e+03]
 [6.085400e+06 8.280000e+02]
 [5.057095e+06 4.350000e+02]]


In [44]:

# Create a pivot table summarizing 'TotalNetValue' by 'Category'
pivot_table = packages_merged.pivot_table(
    values='TotalNetValue', 
    index='Category', 
    aggfunc='sum'
)

print(pivot_table)

           TotalNetValue
Category                
Budget           2470345
Luxury          30615650
Mid-Range       10702390


In [45]:
# Export the data into a CSV
top3_packages.to_csv("Top3_Packages.csv", index=False)
packages_merged.to_csv("PackagePerformance.csv", index=False)
pivot_table.to_csv("category_by_ TotalNetValue.csv")